In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO DO PIPELINE ─────────────────────────────────────────
# Define as referências às tabelas no Unity Catalog usando nomes completamente qualificados (catalog.schema.table). No Databricks Serverless, nomes não qualificados podem falhar quando o catálogo padrão não está configurado.
# TABLE_SOURCE aponta para o dataset Telco Customer Churn (7.043 linhas, 21 colunas) importado diretamente pelo Databricks via interface UI — sem leitura de CSV.

TABLE_SOURCE = "portfolio.default.wa_fn_use_c_telco_customer_churn"
BRONZE_TABLE = "portfolio.default.telco_bronze"

In [0]:
# ─── CÉLULA 2 — INGESTÃO E INSPEÇÃO INICIAL ──────────────────────────────────────
# Lê a tabela-fonte pelo nome registrado no Unity Catalog via spark.table().
# Esta abordagem é preferível a spark.read.format("delta").load(path) pois aproveita o metastore do Databricks e abstrai completamente o caminho físico.
# A inspeção inicial valida que todos os campos foram carregados corretamente antes de qualquer transformação ser aplicada.

df_raw = spark.table(TABLE_SOURCE)
display(df_raw.limit(10))

# Estatísticas descritivas das colunas numéricas principais.
# Permite identificar outliers, ranges esperados e o problema conhecido do campo TotalCharges (vem como string com espaços para clientes com tenure=0).
display(df_raw.describe(["tenure", "MonthlyCharges", "TotalCharges"]))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


summary,tenure,MonthlyCharges,TotalCharges
count,7043,7043,7043
mean,32.37114865824223,64.76169246059922,2283.300440841867
stddev,24.55948102309448,30.09004709767847,2266.7713618831467
min,0,18.25,
max,72,118.75,999.9


In [0]:
# ─── CÉLULA 3 — ESCRITA NA CAMADA BRONZE ─────────────────────────────────────────
# Persiste os dados brutos sem transformações na camada Bronze do Delta Lake.
# Princípio da arquitetura medalhão: Bronze armazena dados exatamente como chegam da fonte, garantindo rastreabilidade e possibilidade de reprocessamento.
# saveAsTable() com Unity Catalog gerencia o caminho físico automaticamente (tabela gerenciada), dispensando especificação de LOCATION.
# mode("overwrite") + overwriteSchema garantem idempotência na re-execução.

(
    df_raw
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"✔ Tabela '{BRONZE_TABLE}' integrada à camada Bronze.")

✔ Tabela 'portfolio.default.telco_bronze' integrada à camada Bronze.


In [0]:
# ─── CÉLULA 4 — VALIDAÇÃO E HISTÓRICO DE VERSÕES ─────────────────────────────────
# Verifica a integridade da tabela recém-criada com três validações:
# 1. DESCRIBE DETAIL: confirma formato Delta, localização gerenciada e tamanho.
# 2. Contagem de linhas: deve ser igual à fonte original (7.043).
# 3. Equivalência de schema: garante que nenhum campo foi perdido ou alterado.
# DESCRIBE HISTORY exibe o Transaction Log do Delta Lake, habilitando Time Travel (consultas a versões anteriores) — recurso fundamental em pipelines de dados.

display(spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}"))

df_check = spark.table(BRONZE_TABLE)
print(f"✔ Linhas na Delta Bronze: {df_check.count()}")
print(f"✔ Schema idêntico ao original: {df_check.schema == df_raw.schema}")

display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c14411c8-493c-42f5-8844-88bb4b9ced00,portfolio.default.telco_bronze,null,abfss://unity-catalog-storage@dbstoragewvatyfu3esnsg.dfs.core.windows.net/7405613588502006/__unitystorage/catalogs/16be19db-f2bb-49bb-8a68-2b240ee8c154/tables/24fd9a84-5ec0-40dc-b5e1-0ec2c749609b,2026-06-04T10:33:45.275Z,2026-06-04T10:33:47Z,List(),List(),1,121517,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


✔ Linhas na Delta Bronze: 7043
✔ Schema idêntico ao original: True


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-04T10:33:47Z,145723392526066,athos.barros@claro.com.br,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3394496807168841),0603-235825-lhas3vyx,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 7043, numOutputBytes -> 121517)",null,Databricks-Runtime/17.3.x-photon-scala2.13
